# 2. Erronka – Smart Environment AI: Mundakako airearen kalitatea

**Modulua:** Adimen Artifizialerako Programazioa · **Zikloa:** IA eta Big Data (FP Santurtzi)

## 1. Sarrera eta testuingurua

Erronka honetan **Bizkaiko Foru Aldundiaren** airearen kalitatearen monitorizazio sareko estazio baten datuekin lan egingo dugu: **Mundakako** estazioarekin. Estazioak egunero neurtzen ditu hainbat kutsatzaile (NO, NOX, NO2, O3, PM10 eta PM2,5), µg/m³ unitatetan.

Gure helburua **Smart Environment AI** soluzio baten oinarria prestatzea da: datuak aztertu (EDA), garbitu eta aurreprozesatzeko *pipeline* bat sortu, gero eredu bat entrenatzeko eta ondoren **Jetson Nano** gailuan exekutatzeko.

Cuaderno honetan hauek egingo ditugu:

1. Datuak kargatu eta egitura aztertu.
2. Azterketa esploratzailea (EDA): estatistikak, banaketak, denbora-bilakaera eta korrelazioak.
3. Garbiketa, balio hutsen tratamendua eta *outlier*-en azterketa.
4. Ezaugarri tenporalak sortu (`Month`, `DayOfWeek`, `DayOfYear`).
5. Aldagaiak eraldatu (estandarizazioa, *one-hot* eta PCA) eta Scikit-Learn *Pipeline* bat sortu, eredu sinple batekin.
6. *Pipeline*-a datu berriekin probatu eta `joblib` bidez gorde (`.pkl`).

## 2. Datuen karga eta egitura

CSV fitxategia `;` bereizlearekin eta `latin1` kodeketarekin irakurtzen dugu (`µ` ikurra egoki ager dadin). Datak `DD/MM/YYYY` formatuan datoz, beraz formatua zehaztu behar dugu `pd.to_datetime` erabiltzean. Gero kronologikoki ordenatuko ditugu, fitxategian atzekoz aurrera baitaude (azken eguna lehenengo).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Grafikoen konfigurazioa
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Datuen karga (bereizlea ';' eta kodeketa 'latin1')
df = pd.read_csv('MUNDAKA.csv', sep=';', encoding='latin1')

# 2. Datak datetime formatura pasa (DD/MM/YYYY) eta ordena kronologikoa
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values('Date').reset_index(drop=True)

# 3. Egituraren laburpena
print("--- DATAFRAME INFO ---")
print(df.info())
print("\n--- BALIO HUTSAK ZUTABEKO ---")
print(df.isnull().sum())
print("\n--- ESTATISTIKA DESKRIBATZAILEAK ---")
display(df.describe())

**Behaketak:**

- 306 errenkada ditugu, **2026-01-01etik 2026-11-02ra** (eguneko neurketak).
- Zutabe guztiak zenbakizkoak dira (`float64`), `Date` izan ezik.
- Balio hutsen kopurua oso desberdina da zutabe batetik bestera: `O3 8h` zutabeak 95 huts ditu (**% 31**), `PM10`-ek 49 (**% 16**) eta gainerakoek 2-4 bakarrik.
- Kontzentrazioak oso baxuak dira (adibidez NO2-ren batezbestekoa ≈ 3 µg/m³), aire garbiko estazio kostalde batean espero daitekeen bezala.

## 3. Azterketa esploratzailea (EDA)

### 3.1. Kutsatzaileen denbora-bilakaera

Hiru kutsatzaile garrantzitsuenen eboluzioa marraztuko dugu: **O3**, **PM10** eta **NO2**. Horrela urtaroen eragina eta pikoak azkar ikusiko ditugu.

In [ ]:
# 1. Kutsatzaile nagusien denbora-bilakaera
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['O3 (µg/m3)'], label='O3', color='green', alpha=0.8)
plt.plot(df['Date'], df['PM10 (µg/m3)'], label='PM10', color='orange', alpha=0.8)
plt.plot(df['Date'], df['NO2 (µg/m3)'], label='NO2', color='red', alpha=0.8)
plt.title('Kutsatzaileen eguneroko bilakaera Mundakan (2026)')
plt.xlabel('Data')
plt.ylabel('Kontzentrazioa (µg/m³)')
plt.legend()
plt.show()

### 3.2. Kutsatzaileen banaketak

Kutsatzaile bakoitzaren balioak nola banatzen diren ikusteko histogramak egingo ditugu.

In [ ]:
# Kutsatzaileen banaketak (histogramak)
kutsatzaile_zutabeak = ['NO (µg/m3)', 'NOX (µg/m3)', 'NO2 (µg/m3)', 'O3 (µg/m3)', 'PM10 (µg/m3)', 'PM2,5 (µg/m3)']
fig, ardatzak = plt.subplots(2, 3, figsize=(14, 7))
for ardatza, zutabea in zip(ardatzak.flatten(), kutsatzaile_zutabeak):
    ardatza.hist(df[zutabea].dropna(), bins=20, color='steelblue', edgecolor='white')
    ardatza.set_title(zutabea.split(' ')[0])
    ardatza.set_xlabel('Kontzentrazioa (µg/m³)')
    ardatza.set_ylabel('Egun kopurua')
plt.suptitle('Kutsatzaileen banaketa Mundakan (2026)')
plt.tight_layout()
plt.show()

Kutsatzaile gehienek balio baxu asko dituzte eta pikoak gutxi (ezkerrera makurtutako banaketak). NO, NOX eta NO2 oso balio baxuetan pilatzen dira, eta O3 da batezbestekoa altuena eta banaketa zabalena duena.

### 3.3. Ozonoaren (O3) eragin estazionala

Grafikoan O3 ikusten da nagusi, eta ez dirudi urtean zehar uniformea denik. Hilabeteko batezbestekoak kalkulatuko ditugu hori zenbakiekin baieztatzeko.

In [ ]:
# Hilabeteko batezbestekoak (Month zutabea beranduago sortuko dugu behin betiko;
# hemen aldi baterako erabiltzen dugu grafikoa egiteko)
hilabete_izenak = ['Urt', 'Ots', 'Mar', 'Api', 'Mai', 'Eka', 'Uzt', 'Abu', 'Ira', 'Urr', 'Aza']
hilabetekoak = df.groupby(df['Date'].dt.month)[['O3 (µg/m3)', 'PM10 (µg/m3)', 'NO2 (µg/m3)']].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hilabete_izenak, hilabetekoak['O3 (µg/m3)'], color='green', alpha=0.75)
ax.set_title('O3-ren hilabeteko batezbestekoa Mundakan (2026)')
ax.set_xlabel('Hilabetea')
ax.set_ylabel('O3 batezbestekoa (µg/m³)')
plt.show()

### 3.4. Korrelazio matrizea

Kutsatzaileen artean erlazioak badauden aztertzeko korrelazio matrizea egingo dugu. Hau garrantzitsua da gero PCA aplikatuko dugulako: aldagai korrelatuak badira, dimentsioa murriztea zentzuzkoa da.

In [ ]:
# 2. Korrelazio matrizea
plt.figure(figsize=(9, 7))
zutabe_zenbakizkoak = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[zutabe_zenbakizkoak].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Kutsatzaileen arteko korrelazio matrizea')
plt.show()

### 3.5. EDA-ren ondorioak

- **O3-ren efektu estazionala:** ozonoa udaberrian eta udan altuagoa da (maiatzean ≈ 80 µg/m³, uztailean ≈ 78 µg/m³) eta udazkenean nabarmen jaisten da (urrian eta azaroan ≈ 55 µg/m³). Zentzuzkoa da: O3 eguzki-erradiazioak eragindako erreakzio fotokimikoetan sortzen da, beraz eguzki-ordu gehiago dauden garaian handiagoa da. Ekainean batezbestekoa apur bat jaisten da, baina hilabete horretan ozono-pikorik handienetako bat ere ageri da (116 µg/m³, ekainaren 24an).
- **NO2 maila oso baxuak:** NO2-ren batezbestekoa ≈ 3 µg/m³ da eta maximoa 8 µg/m³. Mundaka kostaldeko herri txikia da, trafiko eta industria gutxirekin, eta itsasoko aireak kutsatzaileak sakabanatzen laguntzen du. Beraz, NO2 ez da arazo handi bat eremu honetan.
- **Korrelazio handiak:** `PM10` eta `PM2,5` oso lotuta daude (≈ 0,83), PM2,5 PM10-aren zati bat delako. Era berean, `NOX` eta `NO2` (≈ 0,84) eta `NOX` eta `NO` (≈ 0,69) ere lotuta daude, NOX bien batura delako. Informazio errepikatua dago, eta horrek PCA erabiltzea justifikatzen du.
- **O3 eta partikulak:** O3-k PM10-ekin korrelazio moderatua du (≈ 0,44), eta pikoak datu berdinetan batera ageri dira (ekainaren 23-25 inguruan, adibidez). Kausa zehatza ezin dugu datu hauekin ziurtatu, baina kutsadura-episodio puntual baten seinale izan daiteke.
- **Ia korrelaziorik ez O3 eta NOX artean (≈ 0,02):** ez dirudi hemen O3 tokiko NOx-aren mende dagoenik.

## 4. Garbiketa, balio hutsak eta *outlier*-ak

Balio hutsen kopurua 2. atalean ikusi dugu (`isnull().sum()`). Kontuan hartu: `31/08/2026` eta `01/09/2026` egunetan zutabe guztiak hutsik daude (neurketarik gabeko egunak).

### Erabakien justifikazioa

- **`O3 8h` zutabea kentzen dugu.** %31 huts ditu (>%30). Hainbeste daturen falta denez, mediana bidez betetzeak gehienetan asmatutako balioak sortuko lituzke, eta zutabeak ez luke informazio ona emango. Gainera, O3-ren 8 orduko batez bestekoa da, eta `O3` zutabearekin oso lotuta dago (informazio errepikatua), beraz galtzen dugun informazioa oso gutxi da.
- **Gainerako hutsak mediana bidez betetzen ditugu (`SimpleImputer(strategy='median')`).** Hutsak gutxi dira (%0,7-%1,3), eta `PM10` kasuan %16. Mediana batezbestekoa baino hobea da hemen, balio muturrekoek (kutsadura-pikoek) ez dutelako eragiten. Batezbestekoak, aldiz, pikoen aldera mugituko lituzke betetako balioak.
- **Outlier-ak ez ditugu ezabatzen.** Ingurumen-datuetan balio altuak (adibidez O3 = 116 µg/m³ edo PM10 = 35 µg/m³) **kutsadura-pikoak** dira, hau da, benetako gertaerak. Hauek ezabatzeak ereduari ezin izango lioke irakatsi benetan interesatzen zaiguna: kutsadura-episodioak detektatzea. Akats instrumentalen kasuan bakarrik ezabatu beharko genuke, eta hemen ez dago horren zantzurik (balioak fisikoki ahalezkoak dira).

Boxplot batekin *outlier*-ak ikusiko ditugu, ezabatu gabe.

In [ ]:
# Outlier-ak ikusteko boxplot-a (ez dugu ezer ezabatzen)
plt.figure(figsize=(13, 6))
sns.boxplot(data=df[kutsatzaile_zutabeak], orient='h', palette='Set2')
plt.title('Kutsatzaileen banaketa eta outlier-ak (boxplot-a)')
plt.xlabel('Kontzentrazioa (µg/m³)')
plt.ylabel('Kutsatzailea')
plt.show()

Boxplot-ean `PM10`, `PM2,5` eta `NOX` aldagaietan puntu isolatu batzuk ikusten dira. Esan bezala, **mantendu** egingo ditugu: kutsadura-pikoak dira.

## 5. Ezaugarri tenporalak (*Feature Engineering*)

Data bakarrik ez da egokia eredu batentzat, beraz data horretatik hiru aldagai tenporal ateratzen ditugu: `Month` (hilabetea, urtaroaren eragina jasotzeko), `DayOfWeek` (asteko eguna, trafiko-ereduak asteko egunen arabera aldatzen direlako) eta `DayOfYear` (urteko eguna). Ondoren `O3 8h` zutabea kenduko dugu, goian justifikatu bezala.

`DayOfYear` sortzen dugu baina ez dugu ezaugarri gisa sartzen, `Month`-ekin oso lotuta dagoelako eta informazio errepikatua litzatekeelako.

In [ ]:
# 1. Ezaugarri tenporalak sortu
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayOfYear'] = df['Date'].dt.dayofyear

# 2. Hutsune gehiegi duen zutabea kendu (>%30)
if 'O3 8h (µg/m3)' in df.columns:
    df.drop(columns=['O3 8h (µg/m3)'], inplace=True)

# 3. Aldagai prediktoreak (X) eta helburua (y): PM10 aurreikusiko dugu
ezaugarri_zutabeak = ['NO (µg/m3)', 'NOX (µg/m3)', 'NO2 (µg/m3)', 'O3 (µg/m3)', 'PM2,5 (µg/m3)', 'Month', 'DayOfWeek']
helburua = 'PM10 (µg/m3)'
df_eredua = df.dropna(subset=[helburua])   # helburuan hutsik dutenak ezin dira erabili
X = df_eredua[ezaugarri_zutabeak]
y = df_eredua[helburua]

print("Entrenamendurako hautatutako ezaugarriak:")
print(X.head())
print(f"\nX-ren dimentsioak: {X.shape}, y-ren luzera: {len(y)}")

**Oharra:** `PM10` helburu gisa erabiliko dugu (`PM2,5`-ekin lotuta dago, ≈ 0,83), beraz ez dugu ezaugarrien artean sartzen. Helburuan balio hutsa duten errenkadak kendu ditugu.

## 6. Eraldaketa, *Pipeline*-a eta PCA

### 6.1. Zergatik Pipeline bat?

Aurreprozesamendua eta eredua Scikit-Learn-en `Pipeline` bakarrean lotzen ditugu:

1. **`SimpleImputer(strategy='median')`** – aldagai zenbakizkoetako balio hutsak medianaz betetzen ditu.
2. **`StandardScaler()`** – aldagai zenbakizkoak estandarizatzen ditu (batezbestekoa 0, desbideratzea 1). O3 (≈ 70 µg/m³) eta NO (≈ 0,5 µg/m³) eskala oso desberdinetan daude, eta PCA eskalarekiko sentikorra da.
3. **`OneHotEncoder`** – `Month` eta `DayOfWeek` aldagai kategorikotzat hartu eta *one-hot* kodetzen ditu (zenbaki baten ordez, kategoria bakoitzeko zutabe bat), hilabete 12 ez baita hilabete 6 baino "bi aldiz handiagoa".
4. **`PCA(n_components=0.95)`** – dimentsioa murrizten du, **bariantzaren %95** gordez.
5. **`LinearRegression()`** – eredu sinple bat, `PM10` aurreikusteko.

**Pipeline-aren abantaila nagusia *data leakage* saihestea da.** Imputer-ak, scaler-ak eta PCA-k datuetatik ikasten dute (mediana, batezbestekoa, osagaiak). Horregatik datuak lehenik *train*/*test* multzoetan banatzen ditugu eta `fit()` *train* multzoan bakarrik egiten dugu. *Test* multzoan eta datu berrietan `predict()` bakarrik erabiltzen da, *train*-eko parametro berekin.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

zutabe_zenbakizkoak = ['NO (µg/m3)', 'NOX (µg/m3)', 'NO2 (µg/m3)', 'O3 (µg/m3)', 'PM2,5 (µg/m3)']
zutabe_kategorikoak = ['Month', 'DayOfWeek']

# 1. Train/test banaketa (pipeline-a train multzoarekin bakarrik doituko dugu)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Aurreprozesamendua: zenbakizkoak (imputer + scaler) eta kategorikoak (one-hot)
aurreprozesamendua = ColumnTransformer(transformers=[
    ('zenbakizkoak', Pipeline([('imputer', SimpleImputer(strategy='median')),
                               ('scaler', StandardScaler())]), zutabe_zenbakizkoak),
    ('kategorikoak', OneHotEncoder(handle_unknown='ignore', sparse_output=False), zutabe_kategorikoak)
])

# 3. Pipeline osoa: aurreprozesamendua + PCA (%95) + eredua
pipeline_osoa = Pipeline(steps=[
    ('aurreprozesamendua', aurreprozesamendua),
    ('pca', PCA(n_components=0.95)),
    ('eredua', LinearRegression())
])

# 4. Train multzoarekin doitu
pipeline_osoa.fit(X_train, y_train)

# 5. PCA-ren informazioa
pca = pipeline_osoa.named_steps['pca']
print(f"PCA osagai kopurua: {pca.n_components_}")
print(f"Bariantza azaldu metatua: {pca.explained_variance_ratio_.sum():.3f}")

# 6. Test multzoan (datu berriak) probatu
y_iragarpena = pipeline_osoa.predict(X_test)
print(f"\nMAE: {mean_absolute_error(y_test, y_iragarpena):.2f} µg/m³")
print(f"R²: {r2_score(y_test, y_iragarpena):.3f}")

### 6.2. Esportazioa `.pkl` formatuan (`joblib`) eta datu berriekin froga

Pipeline osoa `joblib`-ekin gordetzen dugu. Fitxategi bakarrean gelditzen dira medianak, scaler-aren parametroak, *one-hot* kategoriak, PCA-ren osagaiak eta eredua. Horrela, **Jetson Nano**-n kargatu eta datu berriekin zuzenean erabil daiteke, ezer berriro kalkulatu gabe.

In [ ]:
# 1. Pipeline-a diskoan gorde
joblib.dump(pipeline_osoa, 'pipeline_preprocesamiento_mundaka.pkl')
print("Pipeline-a ondo gorde da: 'pipeline_preprocesamiento_mundaka.pkl'")

# 2. Kargatu eta datu berri batzuekin probatu (test multzoko lehen 3 errenkadak)
pipeline_kargatua = joblib.load('pipeline_preprocesamiento_mundaka.pkl')
datu_berriak = X_test.head(3)
print("\nPM10 aurreikuspena (pipeline kargatua):", pipeline_kargatua.predict(datu_berriak).round(2))
print("PM10 benetako balioak:                 ", y_test.head(3).round(2).values)